In [1]:
import fiftyone as fo

import os

In [2]:
dataset = fo.load_dataset("mp_article_beni")

In [3]:
dataset

Name:        mp_article_beni
Media type:  image
Num samples: 6771
Persistent:  True
Tags:        []
Sample fields:
    id:                                        fiftyone.core.fields.ObjectIdField
    filepath:                                  fiftyone.core.fields.StringField
    tags:                                      fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                                  fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                                fiftyone.core.fields.DateTimeField
    last_modified_at:                          fiftyone.core.fields.DateTimeField
    image_path:                                fiftyone.core.fields.StringField
    sample_type:                               fiftyone.core.fields.StringField
    island:                                    fiftyone.core.fields.StringField
    station:                                   fiftyone.core.fields.StringField

In [4]:
dataset.list_evaluations()

['eval_400x400_beni_hao_mak_tub',
 'eval_400x400_train_test',
 'eval_inst_400x400_beni_hao_mak_tub',
 'eval_inst_400x400_train_test']

In [5]:
session = fo.launch_app(dataset, auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


In [6]:
dataset.list_evaluations()

['eval_400x400_beni_hao_mak_tub',
 'eval_400x400_train_test',
 'eval_inst_400x400_beni_hao_mak_tub',
 'eval_inst_400x400_train_test']

In [15]:
results = dataset.load_evaluation_results("eval_inst_400x400_train_test")

In [16]:
results.plot_pr_curves()

FigureWidget({
    'data': [{'customdata': {'bdata': ('4kBOBkAQ7D8xB+A400HpP8dkV1rvqe' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAA=='),
                             'dtype': 'f8'},
              'hovertemplate': ('<b>class: %{text}</b><br>recal' ... 'customdata:.3f}<extra></extra>'),
              'line': {'color': '#3366CC'},
              'mode': 'lines',
              'name': 'Particles (AP = 0.135)',
              'text': array(['Particles', 'Particles', 'Particles', 'Particles', 'Particles',
                             'Particles', 'Particles', 'Particles', 'Particles', 'Particles',
                             'Particles', 'Particles', 'Particles', 'Particles', 'Particles',
                             'Particles', 'Particles', 'Particles', 'Particles', 'Particles',
                             'Particles', 'Particles', 'Particles', 'Particles', 'Particles',
                             'Particles', 'Particles', 'Particles', 'Particles', 'Particles',
                             'Particle

In [17]:
results.plot_confusion_matrix()

FigureWidget({
    'data': [{'mode': 'markers',
              'opacity': 0.1,
              'type': 'scatter',
              'uid': 'ab64b7dc-dc29-47df-8696-754f5b367aae',
              'x': {'bdata': 'AAECAwABAgMAAQIDAAECAw==', 'dtype': 'i1'},
              'y': {'bdata': 'AAAAAAEBAQECAgICAwMDAw==', 'dtype': 'i1'}},
             {'colorscale': [[0.0, 'rgb(255,245,235)'], [0.125,
                             'rgb(254,230,206)'], [0.25, 'rgb(253,208,162)'],
                             [0.375, 'rgb(253,174,107)'], [0.5, 'rgb(253,141,60)'],
                             [0.625, 'rgb(241,105,19)'], [0.75, 'rgb(217,72,1)'],
                             [0.875, 'rgb(166,54,3)'], [1.0, 'rgb(127,39,4)']],
              'hoverinfo': 'skip',
              'showscale': False,
              'type': 'heatmap',
              'uid': 'f8cf99a7-a97b-4149-83eb-9a3f5680f4e4',
              'z': {'bdata': 'RwBYAIUAAAAAAAAAMgAkAAQAIwAtAEQAagAuAAAAfAA=', 'dtype': 'i2', 'shape': '4, 4'},
              'zmax'

In [18]:
import numpy as np
from fiftyone import ViewField as F
import fiftyone.core.plots as fop

# 1) Build evaluation patches view
eval_patches = dataset.to_evaluation_patches("eval_inst_400x400_train_test")

# 2) All *predictions* that were evaluated (TP or FP)
pred_patches = eval_patches.match(F("type").is_in(["tp", "fp"]))

In [19]:
# Extract confidences and TP/FP flags
confs = pred_patches.values("inst_predictions_400x400_train_test.detections.confidence")
types = pred_patches.values("type")   # "tp" or "fp"

In [20]:
# Flatten because each patch has a single prediction
confs = np.array([c[0] for c in confs])
is_tp = np.array([t == "tp" for t in types])

# 3) Sort by decreasing confidence
order = np.argsort(-confs)
is_tp = is_tp[order]

tp_cum = np.cumsum(is_tp)
fp_cum = np.cumsum(~is_tp)

# 4) Total number of positives = TP + FN (class-agnostic)
num_gt = eval_patches.match(F("type").is_in(["tp", "fn"])).count()

recall = tp_cum / num_gt
precision = tp_cum / (tp_cum + fp_cum)

# 5) Plot a *single* class-agnostic PR curve
fop.plot_pr_curve(precision, recall, label="class-agnostic")


FigureWidget({
    'data': [{'fill': 'tozeroy',
              'hovertemplate': 'recall: %{x:.3f}<br>precision: %{y:.3f}<extra></extra>',
              'line': {'color': '#FF6D04'},
              'mode': 'lines',
              'type': 'scatter',
              'uid': 'efc3d35d-4a82-47d4-915c-b54b3b65f955',
              'x': {'bdata': ('IOAf4B/gXz8g4B/gH+BvPxjoF+gX6H' ... 'g3yDfI1z84yDfIN8jXPzjIN8g3yNc/'),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAA8D8AAAAAAADwPwAAAAAAAP' ... 'aFpelp2T85alw5alzZPydSFxz5Ttk/'),
                    'dtype': 'f8'}}],
    'layout': {'margin': {'b': 0, 'l': 0, 'r': 0, 't': 30},
               'shapes': [{'line': {'dash': 'dash'}, 'type': 'line', 'x0': 0, 'x1': 1, 'y0': 1, 'y1': 0}],
               'template': '...',
               'title': {'text': 'class-agnostic'},
               'xaxis': {'constrain': 'domain', 'range': [0, 1], 'title': {'text': 'Recall'}},
               'yaxis': {'constrain': 'domain',
                    

In [21]:
results.mAP()

0.10561980374511808

In [22]:
def compute_ap(precision, recall):
    # 1) sort by recall just in case (should already be increasing)
    order = np.argsort(recall)
    recall = recall[order]
    precision = precision[order]

    # 2) prepend (0,1) and append (1,0) if you want a closed curve
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))

    # 3) make precision envelope (monotone decreasing)
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])

    # 4) integrate over recall steps
    indices = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[indices + 1] - mrec[indices]) * mpre[indices + 1])
    return ap

agnostic_AP = compute_ap(precision, recall)

In [23]:
agnostic_AP

0.2280849071020455

In [13]:
dataset.save()